# Iceberg tables in the Polaris catalog

In [04b](04b_write_iceberg.ipynb) the catalog was a local folder. Here it is **Apache Polaris**, a REST catalog service from our compose stack. Any engine that speaks the Iceberg REST API can find these tables by name.

- Table files: `warehouse` bucket in RustFS
- Catalog state: `polaris_schema` in PostgreSQL

**Before you start:** `polaris` and `rustfs` must be running, and `Mall_Customers.csv` must be in `/opt/examples/datasets`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

In [ ]:
spark = (
    SparkSession.builder
    .appName("Iceberg Polaris")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0,org.apache.iceberg:iceberg-aws-bundle:1.12.0")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named lakehouse, served by Polaris
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://polaris:8181/api/catalog")
    .config("spark.sql.catalog.lakehouse.warehouse", "lakehouse")
    .config("spark.sql.catalog.lakehouse.credential", "root:s3cr3t")
    .config("spark.sql.catalog.lakehouse.scope", "PRINCIPAL_ROLE:ALL")
    # How Spark reaches the table files in RustFS
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "dataops")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "Ankara06")
    .config("spark.sql.catalog.lakehouse.client.region", "us-east-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()

## 1. Create a namespace and write the table

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.iceberg_db")

In [ ]:
df.writeTo("lakehouse.iceberg_db.mall_customers").using("iceberg").createOrReplace()

In [ ]:
spark.sql("SHOW TABLES IN lakehouse.iceberg_db").show()

In [ ]:
spark.table("lakehouse.iceberg_db.mall_customers").count()

## 2. What does the catalog store?

One row per table, with a pointer to the table's current metadata file.

In [ ]:
import psycopg2

def show_catalog(table):
    conn = psycopg2.connect(host="postgresql", user="postgres", password="Ankara06", dbname="postgres")
    cur = conn.cursor()
    cur.execute("""
        SELECT name, entity_version, internal_properties::jsonb->>'metadata-location'
        FROM polaris_schema.entities
        WHERE name = %s AND type_code = 7
          AND internal_properties::jsonb->>'parent-namespace' = 'iceberg_db'
    """, (table,))
    for row in cur.fetchall():
        print(row)
    conn.close()

In [ ]:
show_catalog("mall_customers")

## 3. Append and look again

The commit writes a new metadata file, and the catalog pointer moves to it. The version number goes up too.

In [ ]:
df.limit(5).writeTo("lakehouse.iceberg_db.mall_customers").append()
spark.table("lakehouse.iceberg_db.mall_customers").count()

In [ ]:
show_catalog("mall_customers")

## 4. Snapshots and time travel

In [ ]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM lakehouse.iceberg_db.mall_customers.snapshots").show(truncate=False)

In [ ]:
first_snapshot = spark.sql(
    "SELECT snapshot_id FROM lakehouse.iceberg_db.mall_customers.snapshots ORDER BY committed_at LIMIT 1"
).collect()[0][0]

spark.read.option("versionAsOf", first_snapshot).table("lakehouse.iceberg_db.mall_customers").count()

## 5. Update and delete

In [ ]:
spark.sql("""
    UPDATE lakehouse.iceberg_db.mall_customers
    SET SpendingScore = SpendingScore + 1
    WHERE Gender = 'Female'
""")

In [ ]:
spark.sql("""
    DELETE FROM lakehouse.iceberg_db.mall_customers
    WHERE Age > 65
""")

In [ ]:
show_catalog("mall_customers")

In [ ]:
spark.stop()